In [1]:
%pip install duckdb

Note: you may need to restart the kernel to use updated packages.


In [2]:
import duckdb ,pandas as pd
from pathlib import Path

ROOT = Path("/Users/shraddha/Documents/DATA_SCIENCE/Shellscope/shellscope")
PARQ = ROOT / "data" / "processed" / "shellscope_companies.parquet"
DB   = ROOT / "data" / "shellscope.duckdb"

con= duckdb.connect(str(DB))
print(duckdb.__version__)

def q(sql):
    return con.sql(sql).df()

1.5.6


In [3]:
q(f"SELECT COUNT(*) AS n FROM read_parquet('{PARQ}')")

,n
0,5704711


In [4]:
con.sql(f"SELECT COUNT(*) AS n FROM read_parquet('{PARQ}')")


┌─────────┐
│    n    │
│  int64  │
├─────────┤
│ 5704711 │
└─────────┘

In [5]:
con.sql(f"CREATE OR REPLACE TABLE companies AS SELECT * FROM read_parquet('{PARQ}')")
print(q("SHOW TABLES"))
print(q("SELECT COUNT(*) AS n FROM companies"))
print(q("DESCRIBE companies")[["column_name", "column_type"]])
summ = q("SUMMARIZE companies")
summ[["column_name", "min", "max", "approx_unique", "null_percentage"]]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

        name
0  companies
         n
0  5704711
                 column_name column_type
0                CompanyName     VARCHAR
1              CompanyNumber     VARCHAR
2    RegAddress.AddressLine1     VARCHAR
3        RegAddress.PostTown     VARCHAR
4        RegAddress.PostCode     VARCHAR
5            CompanyCategory     VARCHAR
6              CompanyStatus     VARCHAR
7          IncorporationDate   TIMESTAMP
8       Accounts.NextDueDate   TIMESTAMP
9   Accounts.AccountCategory     VARCHAR
10         SICCode.SicText_1     VARCHAR
11       ConfStmtNextDueDate   TIMESTAMP
12                  in_scope     BOOLEAN
13         company_age_years      DOUBLE
14          accounts_overdue     BOOLEAN
15          confstmt_overdue     BOOLEAN
16                 is_active     BOOLEAN
17       strike_off_proposal     BOOLEAN
18                  sic_code     VARCHAR
19               sic_section     VARCHAR
20            postcode_clean     VARCHAR
21             postcode_area     VARCHAR
22       

,column_name,min,max,approx_unique,null_percentage
0,CompanyName,! LTD,€ RECYCLING LIMITED,6848663,0.00
1,CompanyNumber,00000086,ZC000206,5055648,0.00
2,RegAddress.AddressLine1,!8 QUAY STREET,‘THE STUDIO’ AT UNIT 2 LOW FARM,1761196,1.11
3,RegAddress.PostTown,21 OLD CHANNEL ROAD,ŞIŞLI,49599,1.66
4,RegAddress.PostCode,IP28 7DE,ZUG 6302,1126554,1.60
5,CompanyCategory,Charitable Incorporated Organisation,United Kingdom Societas,26,0.00
6,CompanyStatus,ADMINISTRATION ORDER,Voluntary Arrangement,16,0.00
7,IncorporationDate,1327-01-01 00:00:00,2026-10-01 00:00:00,33276,0.00
8,Accounts.NextDueDate,1876-11-01 00:00:00,2028-09-30 00:00:00,7851,3.04
9,Accounts.AccountCategory,ACCOUNTS TYPE NOT AVAILABLE,UNAUDITED ABRIDGED,15,0.00


In [6]:
q("SELECT COUNT(*) AS n FROM companies")

,n
0,5704711


In [7]:
q("""
SELECT CompanyStatus,
       COUNT(*) AS n,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM companies
GROUP BY CompanyStatus
ORDER BY n DESC
""")

,CompanyStatus,n,pct
0,Active,5166057,90.56
1,Active - Proposal to Strike off,423006,7.42
2,Liquidation,108330,1.90
3,In Administration,3709,0.07
4,Live but Receiver Manager on at least one charge,2156,0.04
5,Voluntary Arrangement,583,0.01
6,In Administration/Administrative Receiver,398,0.01
7,RECEIVERSHIP,182,0.00
8,ADMINISTRATION ORDER,121,0.00
9,ADMINISTRATIVE RECEIVER,104,0.00


In [8]:
q("""
SELECT "RegAddress.PostTown" AS town, COUNT(*) AS n
FROM companies
WHERE "RegAddress.PostTown" IS NOT NULL
GROUP BY town
ORDER BY n DESC
LIMIT 10
""")

,town,n
0,LONDON,1111178
1,MANCHESTER,112616
2,BIRMINGHAM,99065
3,GLASGOW,75393
4,EDINBURGH,61238
5,BRISTOL,58631
6,LEEDS,54536
7,CARDIFF,53744
8,LIVERPOOL,49485
9,LEICESTER,47060


In [9]:
q("""
SELECT COUNT(*) As n_2025
From companies
WHERE IncorporationDate >= DATE '2025-01-01' AND IncorporationDate < DATE '2026-01-01'
""")

,n_2025
0,715974


In [10]:
q("""
SELECT CompanyCategory,
       COUNT(*) AS n,
       ROUND(AVG(date_diff('year', IncorporationDate, DATE '2026-10-01')), 2) AS avg_age_yearcount,
       ROUND(AVG(date_diff('day',  IncorporationDate, DATE '2026-10-01') / 365.25), 2) AS avg_age_true,
       ROUND(MEDIAN(date_diff('day', IncorporationDate, DATE '2026-10-01') / 365.25), 2) AS median_age
FROM companies
GROUP BY CompanyCategory
ORDER BY n DESC
LIMIT 8
""")

,CompanyCategory,n,avg_age_yearcount,avg_age_true,median_age
0,Private Limited Company,5276662,8.84,9.11,5.75
1,"PRI/LTD BY GUAR/NSC (Private, limited by guara...",118855,15.82,16.07,11.41
2,Limited Partnership,61086,14.92,15.16,12.25
3,Limited Liability Partnership,50171,9.38,9.64,8.82
4,Community Interest Company,45328,4.17,4.43,2.69
5,Charitable Incorporated Organisation,40659,5.75,6.00,5.93
6,"PRI/LBG/NSC (Private, Limited by guarantee, no...",35840,17.29,17.55,16.05
7,Overseas Entity,30207,3.11,3.46,3.68


In [12]:
q("""
SELECT "RegAddress.PostTown" AS town, COUNT(*) AS n
FROM companies
GROUP BY town
HAVING n >= 10000
ORDER BY n DESC
""")

,town,n
0,LONDON,1111178
1,MANCHESTER,112616
2,BIRMINGHAM,99065
3,NaN,94569
4,GLASGOW,75393
...,...,...
81,FAREHAM,10359
82,DARTFORD,10248
83,WIGAN,10143
84,MIDDLESBROUGH,10069


In [13]:
q("""
SELECT CompanyNumber, CompanyName, CompanyCategory, IncorporationDate
FROM companies
ORDER BY IncorporationDate ASC
LIMIT 10
""")

,CompanyNumber,CompanyName,CompanyCategory,IncorporationDate
0,RC000941,THE WARDENS AND THE COMMONALTY OF THE MYSTERY ...,Royal Charter Company,1327-01-01
1,RC000913,THE WORSHIPFUL COMPANY OF CUTLERS,Royal Charter Company,1416-12-04
2,RC000907,THE WORSHIPFUL COMPANY OF BREWERS,Royal Charter Company,1437-02-22
3,RC000919,THE WORSHIPFUL COMPANY OF CORDWAINERS,Royal Charter Company,1439-04-26
4,RC000935,THE WORSHIPFUL COMPANY OF TALLOW CHANDLERS,Royal Charter Company,1462-03-08
5,RC000934,THE WORSHIPFUL COMPANY OF IRONMONGERS,Royal Charter Company,1463-03-20
6,RC000911,THE DYERS' COMPANY,Royal Charter Company,1471-12-02
7,RC000915,THE WORSHIPFUL COMPANY OF CARPENTERS,Royal Charter Company,1477-07-07
8,RC000933,THE WORSHIPFUL COMPANY OF COOKS OF LONDON,Royal Charter Company,1482-07-11
9,RC000931,THE WORSHIPFUL COMPANY OF PLAISTERERS,Royal Charter Company,1501-03-10


In [14]:
q("""
SELECT COUNT(*)                         AS rows,
       COUNT(postcode_clean)            AS with_postcode,
       COUNT(*) - COUNT(postcode_clean) AS missing_postcode,
       COUNT(DISTINCT postcode_clean)   AS distinct_postcodes,
       COUNT(DISTINCT postcode_area)    AS distinct_areas
FROM companies
""")

,rows,with_postcode,missing_postcode,distinct_postcodes,distinct_areas
0,5704711,5613232,91479,979416,276


In [15]:
q("""
SELECT CASE
         WHEN company_age_years < 2  THEN '1: <2 years'
         WHEN company_age_years < 5  THEN '2: 2-5 years'
         WHEN company_age_years < 10 THEN '3: 5-10 years'
         ELSE '4: 10+ years'
       END AS age_band,
       COUNT(*) AS n
FROM companies
GROUP BY age_band
ORDER BY age_band
""")

,age_band,n
0,1: <2 years,1428903
1,2: 2-5 years,1147499
2,3: 5-10 years,1292875
3,4: 10+ years,1835434


In [16]:
q("""
SELECT CASE WHEN glasgow THEN 'Glasgow (G)' ELSE 'Rest of UK' END AS region,
       COUNT(*) AS n_active_in_scope,
       ROUND(100.0 * AVG(CAST(accounts_overdue AS INTEGER)), 2) AS overdue_pct
FROM companies
WHERE in_scope AND is_active
GROUP BY region

UNION ALL

SELECT 'ALL', COUNT(*), ROUND(100.0 * AVG(CAST(accounts_overdue AS INTEGER)), 2)
FROM companies
WHERE in_scope AND is_active
""")

,region,n_active_in_scope,overdue_pct
0,Rest of UK,4923635,1.88
1,Glasgow (G),69455,2.04
2,ALL,4993090,1.88


In [17]:
## Part C: Data dictionary
desc = q("DESCRIBE companies")[["column_name", "column_type"]]
summ = q("SUMMARIZE companies")[["column_name", "approx_unique", "null_percentage"]]
ex = q("SELECT * FROM companies WHERE CompanyStatus = 'Active' AND postcode_clean IS NOT NULL LIMIT 1").T
ex = ex.reset_index(); ex.columns = ["column_name", "example"]
dd = desc.merge(summ, on="column_name").merge(ex, on="column_name")
dd

,column_name,column_type,approx_unique,null_percentage,example
0,CompanyName,VARCHAR,6848663,0.00,! LTD
1,CompanyNumber,VARCHAR,5055648,0.00,08209948
2,RegAddress.AddressLine1,VARCHAR,1761196,1.11,9 PRINCES SQUARE
3,RegAddress.PostTown,VARCHAR,49599,1.66,HARROGATE
4,RegAddress.PostCode,VARCHAR,1126554,1.60,HG1 1ND
5,CompanyCategory,VARCHAR,26,0.00,Private Limited Company
6,CompanyStatus,VARCHAR,16,0.00,Active
7,IncorporationDate,TIMESTAMP,33276,0.00,2012-09-11 00:00:00
8,Accounts.NextDueDate,TIMESTAMP,7851,3.04,2027-06-30 00:00:00
9,Accounts.AccountCategory,VARCHAR,15,0.00,DORMANT


In [18]:
MEANING = {
 "CompanyName": "Registered company name",
 "CompanyNumber": "Unique company ID (8 characters; prefixes e.g. SC = Scotland, NI = Northern Ireland, OC = LLP)",
 "RegAddress.AddressLine1": "First line of the registered office address",
 "RegAddress.PostTown": "Post town of the registered office",
 "RegAddress.PostCode": "Postcode of the registered office (raw)",
 "CompanyCategory": "Legal type (private limited, LLP, limited partnership, ...)",
 "CompanyStatus": "Current status (Active, Proposal to Strike off, Liquidation, ...)",
 "IncorporationDate": "Date the company was set up",
 "Accounts.NextDueDate": "Date the next annual accounts are due",
 "Accounts.AccountCategory": "Type of the last accounts filed (e.g. micro-entity, dormant, no accounts filed)",
 "SICCode.SicText_1": "First industry code with its description",
 "ConfStmtNextDueDate": "Date the next confirmation statement is due",
 "in_scope": "Derived: company must file accounts at Companies House (Part 1 scope rule)",
 "company_age_years": "Derived: years from IncorporationDate to the snapshot date (2026-10-01)",
 "accounts_overdue": "Derived: Accounts.NextDueDate is before the snapshot date",
 "confstmt_overdue": "Derived: ConfStmtNextDueDate is before the snapshot date",
 "is_active": "Derived: company is active (see the exact rule in 01_explore)",
 "strike_off_proposal": "Derived: status is 'Active - Proposal to Strike off'",
 "sic_code": "Derived: 5-digit SIC code taken from SICCode.SicText_1",
 "sic_section": "Derived: SIC 2007 section letter A-U, plus 'None supplied' and 'Dormant'",
 "postcode_clean": "Derived: postcode in upper case without spaces",
 "postcode_area": "Derived: letters before the first digit of the postcode (G = Glasgow)",
 "glasgow": "Derived: postcode_area == 'G'",
}
QUIRKS = {
 "CompanyNumber": "Must stay TEXT: numeric types drop the leading zeros. Raw CSV header had a leading space.",
 "RegAddress.AddressLine1": "Messy: '3RD FLOOR' alone (many buildings), 'C/O ...', ST vs STREET; ~1.1% missing.",
 "RegAddress.PostTown": "~1.7% missing.",
 "RegAddress.PostCode": "~1.6% missing; inconsistent spacing (use postcode_clean).",
 "CompanyCategory": "92.5% are Private Limited Company.",
 "CompanyStatus": "No 'Dissolved' status: live register only, so closed companies are missing (survivorship bias).",
 "IncorporationDate": "Raw format dd/mm/yyyy. Min 1327-01-01 (royal charter bodies). Oct 2026 incomplete (snapshot day).",
 "accounts_overdue": "Only meaningful for in_scope AND is_active companies.",
 "sic_section": "'None supplied' (247,820 rows) and 'Dormant' (104,507) are not industries; 'None supplied' has a 10.4% overdue rate.",
 "company_age_years": "Right-skewed; use the median or a log scale.",
}

lines = ["# ShellScope data dictionary",
         "",
         "Source: Companies House Basic Company Data (live register), snapshot 2026-10-01. "
         "Table `companies` in `data/shellscope.duckdb` (5,704,711 rows).",
         "",
         "| column | type | example | approx. distinct | % missing | meaning | quirks |",
         "|---|---|---|---|---|---|---|"]
for r in dd.itertuples(index=False):
    lines.append(f"| `{r.column_name}` | {r.column_type} | {str(r.example)[:40]} | {r.approx_unique:,} | "
                 f"{r.null_percentage} | {MEANING.get(r.column_name, '')} | {QUIRKS.get(r.column_name, '')} |")

(ROOT / "docs").mkdir(exist_ok=True)
(ROOT / "docs" / "data_dictionary.md").write_text("\n".join(lines))
print("\n".join(lines[:10]))

# ShellScope data dictionary

Source: Companies House Basic Company Data (live register), snapshot 2026-10-01. Table `companies` in `data/shellscope.duckdb` (5,704,711 rows).

| column | type | example | approx. distinct | % missing | meaning | quirks |
|---|---|---|---|---|---|---|
| `CompanyName` | VARCHAR | ! LTD | 6,848,663 | 0.0 | Registered company name |  |
| `CompanyNumber` | VARCHAR | 08209948 | 5,055,648 | 0.0 | Unique company ID (8 characters; prefixes e.g. SC = Scotland, NI = Northern Ireland, OC = LLP) | Must stay TEXT: numeric types drop the leading zeros. Raw CSV header had a leading space. |
| `RegAddress.AddressLine1` | VARCHAR | 9 PRINCES SQUARE | 1,761,196 | 1.11 | First line of the registered office address | Messy: '3RD FLOOR' alone (many buildings), 'C/O ...', ST vs STREET; ~1.1% missing. |
| `RegAddress.PostTown` | VARCHAR | HARROGATE | 49,599 | 1.66 | Post town of the registered office | ~1.7% missing. |


In [19]:
con.close()